### V_CRG_STUDENT_COURSE — Preprocessing V5

Scope:
- Work only on `V_CRG_STUDENT_COURSE`.
- Load raw Parquet.
- Preserve raw data and original ID values.
- Create clean attempts, normalized attempts, and current student-course status.
- Do not build models, course difficulty, or recommendation logic.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import (
    RAW_DIR,
    CLEAN_DIR,
    FEATURES_DIR as BASE_FEATURES_DIR,
    REPORTS_DIR as BASE_REPORTS_DIR,
    ensure_dir,
)
from src.cleaning_utils import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

VERSION = "v5"

RAW_PATH = RAW_DIR / "v_crg_student_course_raw.parquet"
PREPROCESSED_DIR = ensure_dir(CLEAN_DIR / "V_CRG_STUDENT_COURSE")
FEATURES_DIR = ensure_dir(BASE_FEATURES_DIR / "V_CRG_STUDENT_COURSE")
REPORTS_DIR = ensure_dir(BASE_REPORTS_DIR / "V_CRG_STUDENT_COURSE")

print("VERSION:", VERSION)
print("RAW_PATH:", RAW_PATH)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("FEATURES_DIR:", FEATURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

assert RAW_PATH.exists(), f"Raw file not found: {RAW_PATH}"


### 1. Load raw Parquet file

This step loads the raw V5 Parquet file.  
No values are changed here.

In [ ]:
df_raw_loaded_v5 = pd.read_parquet(RAW_PATH)

print("Raw loaded shape:", df_raw_loaded_v5.shape)
display(df_raw_loaded_v5.head())
display(df_raw_loaded_v5.dtypes)

In [ ]:
assert isinstance(df_raw_loaded_v5, pd.DataFrame)
assert len(df_raw_loaded_v5) > 0, "Raw dataframe is empty."

print("Validation passed: raw Parquet loaded successfully.")

### 2. Standardize column names and validate required columns

Only column names are standardized to lowercase.  
Raw values remain unchanged.

In [ ]:
df_raw_v5 = df_raw_loaded_v5.copy()

df_raw_v5.columns = (
    df_raw_v5.columns
    .astype(str)
    .str.strip()
    .str.lower()
)

required_raw_columns_v5 = [
    "student_course_id",
    "student_id",
    "course_id",
    "part_id",
    "grade_id",
    "final_mark",
    "points",
    "finish_status",
    "course_name_sl",
    "study_mode",
    "degree_id",
    "degree_name_sl",
    "faculty_id",
    "course_credits",
    "active",
]

missing_required_columns_v5 = sorted(set(required_raw_columns_v5) - set(df_raw_v5.columns))
extra_columns_v5 = sorted(set(df_raw_v5.columns) - set(required_raw_columns_v5))

print("Missing required columns:", missing_required_columns_v5)
print("Extra source columns:", extra_columns_v5)

assert len(missing_required_columns_v5) == 0, f"Missing columns: {missing_required_columns_v5}"

df_raw_v5 = df_raw_v5[required_raw_columns_v5].copy()

print("Raw selected shape:", df_raw_v5.shape)
display(df_raw_v5.head())

In [ ]:
assert list(df_raw_v5.columns) == required_raw_columns_v5
assert df_raw_v5.shape[1] == len(required_raw_columns_v5)
assert "is_last_try" not in df_raw_v5.columns, "is_last_try should not be used in V5 workflow."

print("Validation passed: all required columns exist and selected.")

### 3. save Raw profile report

This report gives a quick overview of row count, nulls, unique values, and dtype per column.

In [ ]:
raw_profile_report_v5 = pd.DataFrame({
    "column": df_raw_v5.columns,
    "dtype": [str(df_raw_v5[col].dtype) for col in df_raw_v5.columns],
    "row_count": len(df_raw_v5),
    "non_null_count": [int(df_raw_v5[col].notna().sum()) for col in df_raw_v5.columns],
    "null_count": [int(df_raw_v5[col].isna().sum()) for col in df_raw_v5.columns],
    "null_ratio": [float(df_raw_v5[col].isna().mean()) for col in df_raw_v5.columns],
    "unique_count": [int(df_raw_v5[col].nunique(dropna=True)) for col in df_raw_v5.columns],
})

raw_profile_report_path_v5 = REPORTS_DIR / "raw_profile_report_v5.csv"
raw_profile_report_v5.to_csv(raw_profile_report_path_v5, index=False, encoding="utf-8-sig")

display(raw_profile_report_v5)
print("Saved:", raw_profile_report_path_v5)

In [ ]:
assert raw_profile_report_path_v5.exists()
assert len(raw_profile_report_v5) == len(required_raw_columns_v5)

print("Validation passed: raw profile report saved.")

## 4. Save Null report

This report shows missing values per column.  
It is important before deciding archive/drop or critical issues.

In [ ]:
null_report_v5 = (
    df_raw_v5
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "null_count"})
)

null_report_v5["row_count"] = len(df_raw_v5)
null_report_v5["null_ratio"] = null_report_v5["null_count"] / len(df_raw_v5)

null_report_v5 = null_report_v5.sort_values("null_count", ascending=False).reset_index(drop=True)

null_report_path_v5 = REPORTS_DIR / "null_report_v5.csv"
null_report_v5.to_csv(null_report_path_v5, index=False, encoding="utf-8-sig")

display(null_report_v5)
print("Saved:", null_report_path_v5)

In [ ]:
assert null_report_path_v5.exists()
assert set(null_report_v5["column"]) == set(required_raw_columns_v5)

print("Validation passed: null report saved.")

In [ ]:
plot_df = null_report_v5.sort_values("null_ratio", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["column"], plot_df["null_ratio"])
plt.title("Null Ratio by Column — V_CRG_STUDENT_COURSE V5")
plt.xlabel("Null Ratio")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

### 5. Save Finish status distribution

`finish_status` is the only official source for pass/fail outcome.  
`final_mark >= 50` must not be used as pass logic.

In [ ]:
finish_status_distribution_v5 = (
    df_raw_v5["finish_status"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("<NULL>")
    .value_counts(dropna=False)
    .reset_index()
)

finish_status_distribution_v5.columns = ["finish_status", "count"]
finish_status_distribution_v5["row_count"] = len(df_raw_v5)
finish_status_distribution_v5["ratio"] = finish_status_distribution_v5["count"] / len(df_raw_v5)

finish_status_distribution_path_v5 = REPORTS_DIR / "finish_status_distribution_v5.csv"
finish_status_distribution_v5.to_csv(
    finish_status_distribution_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(finish_status_distribution_v5)
print("Saved:", finish_status_distribution_path_v5)

In [ ]:
assert finish_status_distribution_path_v5.exists()
assert finish_status_distribution_v5["count"].sum() == len(df_raw_v5)

print("Validation passed: finish_status distribution saved.")

In [ ]:
plot_df = finish_status_distribution_v5.sort_values("count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["finish_status"], plot_df["count"])
plt.title("finish_status Distribution — V5")
plt.xlabel("Rows")
plt.ylabel("finish_status")
plt.tight_layout()
plt.show()

## 6. Numeric validation without stopping the notebook

Rules:
- `final_mark` should be integer-like.
- `course_credits` may contain unexpected fractional values; do not round automatically.
- `points` is float and remains raw.

Important:
- If `course_credits` has fractional values, report them and continue.
- Do not manually fill or modify `course_credits`.


In [ ]:
numeric_validation_report_v5 = pd.DataFrame([
    integer_like_report(df_raw_v5, "final_mark"),
    integer_like_report(df_raw_v5, "course_credits"),
])

points_numeric_v5 = pd.to_numeric(df_raw_v5["points"], errors="coerce")

points_report_v5 = pd.DataFrame([{
    "column": "points",
    "source_dtype": str(df_raw_v5["points"].dtype),
    "non_null_count": int(df_raw_v5["points"].notna().sum()),
    "numeric_count": int(points_numeric_v5.notna().sum()),
    "non_numeric_or_null_count": int(df_raw_v5["points"].shape[0] - points_numeric_v5.notna().sum()),
    "fractional_count": np.nan,
    "fractional_ratio": np.nan,
}])

numeric_validation_report_v5 = pd.concat(
    [numeric_validation_report_v5, points_report_v5],
    ignore_index=True
)

numeric_validation_report_path_v5 = REPORTS_DIR / "numeric_validation_report_v5.csv"
numeric_validation_report_v5.to_csv(
    numeric_validation_report_path_v5,
    index=False,
    encoding="utf-8-sig"
)

display(numeric_validation_report_v5)
print("Saved:", numeric_validation_report_path_v5)


In [ ]:
final_mark_fractional = numeric_validation_report_v5.loc[
    numeric_validation_report_v5["column"].eq("final_mark"),
    "fractional_count"
].iloc[0]

course_credits_fractional = numeric_validation_report_v5.loc[
    numeric_validation_report_v5["column"].eq("course_credits"),
    "fractional_count"
].iloc[0]

assert final_mark_fractional == 0, "final_mark has fractional values."

if course_credits_fractional > 0:
    print("WARNING: course_credits has fractional values.")
    print("These rows will be reported and flagged. No rounding will be applied.")
else:
    print("course_credits is integer-like.")

assert numeric_validation_report_path_v5.exists()

print("Validation passed: final_mark is integer-like. course_credits inspected.")